# Data Acquisition & Merge

### Importing os
This step makes sure the project directory is connected to the notebooks

In [1]:
import os
os.chdir("..")  # Go up one level from notebooks/ to project root
print(os.getcwd())

C:\Users\hanis\Flight\flight-delay-prediction


### Overview

Load and merge 5 months of BTS Reporting Carrier On-Time Performance data 
(August–December 2024). The raw CSV files are stored in `data/raw/`; 
the merged dataset is saved as parquet in `data/processed/` for efficient 
reloading and consistent dtypes.


In [2]:
import glob
import pandas as pd

files = sorted(glob.glob("data/raw/*.csv"))
print(files) #in case you want to see what files are present

dfs = [pd.read_csv(f, low_memory=False) for f in files]

# Verify columns match
cols = [tuple(d.columns) for d in dfs]
assert all(c == cols[0] for c in cols), "Columns don't match across files!"

# CONCAT FIRST
df = pd.concat(dfs, ignore_index=True)

# Then drop phantom column
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

# Then save
df.to_parquet("data/processed/flights.parquet") 

['data/raw\\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_10.csv', 'data/raw\\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_11.csv', 'data/raw\\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_12.csv', 'data/raw\\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_8.csv', 'data/raw\\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_9.csv']


## Load & Merge Strategy
- Read each CSV with `low_memory=False` to avoid dtype inconsistencies
- Verify column consistency across files before concatenating
- Drop phantom "Unnamed" columns created by trailing commas in CSV exports
- Save as parquet to preserve dtypes and reduce file size

In [3]:
# Then inspect
print("Shape: ", df.shape)
print(df.columns.tolist())
df.head()

Shape:  (2983129, 109)
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'FirstDe

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum
0,2024,4,10,29,2,2024-10-29,UA,19977,UA,N37504,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,4,10,29,2,2024-10-29,UA,19977,UA,N69824,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,4,10,29,2,2024-10-29,UA,19977,UA,N37297,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024,4,10,29,2,2024-10-29,UA,19977,UA,N47569,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024,4,10,29,2,2024-10-29,UA,19977,UA,N69826,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Result
2.98M flights, 109 columns. Ready for EDA.